<a href="https://colab.research.google.com/github/simon-clematide/colab-notebooks-for-teaching/blob/main/xor_Logistic_Regression_With_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The XOR Problem: From Linear Limits to Deep Learning with PyTorch

The **exclusive OR (XOR)** problem is the classic historical challenge in machine learning (Minsky & Papert, 1969).
XOR outputs `1` if and only if exactly one of its binary inputs is `1`:

| $x_1$ | $x_2$ | $y$ (XOR) |
| :---: | :---: | :-------: |
| 0     | 0     | 0         |
| 0     | 1     | 1         |
| 1     | 0     | 1         |
| 1     | 1     | 0         |

In this notebook, we explore three stages:
1. **Linear Logistic Regression:** Why gradient descent on a linear model provably fails on XOR.
2. **Feature Engineering:** How manually adding an interaction feature ($x_1 \cdot x_2$) makes the problem linearly separable.
3. **Multi-Layer Perceptron (MLP):** How a neural network with a hidden layer and non-linear activation learns the required representation automatically.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Ensure reproducibility
torch.manual_seed(42)

# XOR Dataset
X = torch.tensor([[0.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 0.0],
                  [1.0, 1.0]])
y = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

def plot_decision_boundary(model_fn, title, X_data=X, y_data=y):
    """Helper to plot the 2D decision boundary and probability surface."""
    grid_x, grid_y = torch.meshgrid(
        torch.linspace(-0.2, 1.2, 200),
        torch.linspace(-0.2, 1.2, 200),
        indexing='xy'
    )
    grid_points = torch.stack([grid_x.flatten(), grid_y.flatten()], dim=1)
    
    with torch.no_grad():
        preds = model_fn(grid_points).reshape(grid_x.shape)
    
    plt.figure(figsize=(5, 4))
    cs = plt.contourf(grid_x.numpy(), grid_y.numpy(), preds.numpy(), levels=20, cmap="coolwarm", alpha=0.7)
    plt.colorbar(cs, label="$P(y=1)$")
    plt.contour(grid_x.numpy(), grid_y.numpy(), preds.numpy(), levels=[0.5], colors="black", linewidths=2)
    
    # Plot the 4 XOR data points
    for pt, label in zip(X_data, y_data):
        color = "red" if label.item() == 1.0 else "blue"
        marker = "o" if label.item() == 1.0 else "s"
        plt.scatter(pt[0].item(), pt[1].item(), c=color, marker=marker, s=120, edgecolors="k", zorder=5)
        
    plt.title(title)
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$")
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.show()

## Part 1: Linear Logistic Regression with PyTorch Autograd

We set up a simple linear model: $\hat{y} = \sigma(\mathbf{x} \mathbf{w} + b)$.  
We use PyTorch tensors with `requires_grad=True` to observe automatic differentiation directly.

In [ ]:
# Weights: 2 inputs -> 1 output, plus bias
torch.manual_seed(1)
w = torch.randn(2, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

lr = 1.0
losses_linear = []

print("Training Linear Logistic Regression on XOR:")
for epoch in range(101):
    # Forward pass
    logits = X @ w + b
    preds = torch.sigmoid(logits)
    
    # Binary Cross-Entropy Loss
    loss = nn.functional.binary_cross_entropy(preds, y)
    losses_linear.append(loss.item())
    
    # Backward pass (PyTorch autograd computes dloss/dw and dloss/db)
    loss.backward()
    
    # Gradient descent update step
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
        
    if epoch % 25 == 0:
        acc = ((preds > 0.5) == y).float().mean().item() * 100
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f} | Accuracy: {acc:.0f}%")

print("\nFinal predictions on XOR inputs:")
for i in range(4):
    print(f"x = {X[i].tolist()} -> target: {int(y[i].item())}, pred: {preds[i].item():.3f}")

# Plot the resulting linear boundary
plot_decision_boundary(lambda pts: torch.sigmoid(pts @ w + b), "Linear Model on XOR (Fails: Acc ~ 50%)")

**Observation:**  
The linear model cannot achieve better than **50% accuracy**. The loss stagnates near **0.693** ($-\ln(0.5)$), predicting $\approx 0.5$ for all points. A single straight line cannot separate the two classes.

## Part 2: Feature Engineering ($x_1 \cdot x_2$)

If the problem is not linearly separable in the raw input space $\mathbb{R}^2$, we can engineer a non-linear interaction feature:
$$\phi(\mathbf{x}) = [x_1, \; x_2, \; x_1 \cdot x_2]$$

Notice what $x_1 \cdot x_2$ does:
- For $(0,0)$, $(0,1)$, $(1,0)$, the product is `0`.
- For $(1,1)$, the product is `1`.

Now the data becomes linearly separable in $\mathbb{R}^3$!

In [ ]:
# Expand input with the interaction feature x1 * x2
x1_x2 = (X[:, 0] * X[:, 1]).unsqueeze(1)
X_engineered = torch.cat([X, x1_x2], dim=1)  # shape: (4, 3)

# Weights for 3 features
torch.manual_seed(1)
w_fe = torch.zeros(3, 1, requires_grad=True)
b_fe = torch.zeros(1, requires_grad=True)

lr = 2.0
losses_fe = []

print("Training Linear Model with Feature Engineering [x1, x2, x1*x2]:")
for epoch in range(101):
    logits = X_engineered @ w_fe + b_fe
    preds = torch.sigmoid(logits)
    loss = nn.functional.binary_cross_entropy(preds, y)
    losses_fe.append(loss.item())
    
    loss.backward()
    with torch.no_grad():
        w_fe -= lr * w_fe.grad
        b_fe -= lr * b_fe.grad
        w_fe.grad.zero_()
        b_fe.grad.zero_()
        
    if epoch % 25 == 0:
        acc = ((preds > 0.5) == y).float().mean().item() * 100
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f} | Accuracy: {acc:.0f}%")

print("\nLearned Weights:", w_fe.squeeze().tolist(), "Bias:", b_fe.item())

# Decision boundary in the original 2D input space
def model_fe(pts):
    interaction = (pts[:, 0] * pts[:, 1]).unsqueeze(1)
    expanded = torch.cat([pts, interaction], dim=1)
    return torch.sigmoid(expanded @ w_fe + b_fe)

plot_decision_boundary(model_fe, "Feature Engineered Model: Non-linear Boundary in Input Space")

## Part 3: Learning the Representation with an MLP

Instead of hand-crafting features, can a neural network **learn the representation** on its own?

We build a 2-layer Feed-Forward Neural Network (Multilayer Perceptron):
- **Input layer:** 2 neurons ($x_1, x_2$)
- **Hidden layer:** 2 neurons with **ReLU** (or Sigmoid) activation
- **Output layer:** 1 neuron with Sigmoid output

In [ ]:
# Define a tiny 2-layer MLP
torch.manual_seed(42)
mlp = nn.Sequential(
    nn.Linear(2, 2),
    nn.ReLU(),
    nn.Linear(2, 1),
    nn.Sigmoid()
)

optimizer = torch.optim.Adam(mlp.parameters(), lr=0.1)
criterion = nn.BCELoss()

losses_mlp = []
print("Training 2-layer MLP on XOR:")
for epoch in range(301):
    optimizer.zero_grad()
    preds = mlp(X)
    loss = criterion(preds, y)
    loss.backward()
    optimizer.step()
    losses_mlp.append(loss.item())
    
    if epoch % 50 == 0:
        acc = ((preds > 0.5) == y).float().mean().item() * 100
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f} | Accuracy: {acc:.0f}%")

print("\nFinal MLP predictions on XOR inputs:")
for i in range(4):
    print(f"x = {X[i].tolist()} -> target: {int(y[i].item())}, pred: {preds[i].item():.4f}")

plot_decision_boundary(lambda pts: mlp(pts), "2-Layer MLP on XOR (Learned Representation)")

## Summary & Conceptual Takeaways

1. **Linear Limits:** A single-layer linear model cannot solve XOR because the two classes cannot be separated by a hyper-plane.
2. **Feature Engineering vs. Deep Learning:**
   - In classical ML, we manually engineer transformations ($x_1 \cdot x_2$).
   - In deep learning, hidden layers equipped with non-linear activation functions (ReLU, Sigmoid) learn to transform the input space into a new coordinate system where the data *becomes* linearly separable.
3. **Autodiff in PyTorch:** The training loop remains identical across linear models and deep neural networks: `forward() -> loss -> loss.backward() -> step()`.